## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [234]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [ ]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kabkot"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")

#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [236]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [237]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Diskrepansi Prov vs KabKot")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'provinsi', 'adhb_pdrb_q1-26', 'adhb_pdrb_q2-26', 'adhb_pkrt_q1-26', 'adhb_pkrt_q2-26', 'adhb_pklnprt_q1-26', 'adhb_pklnprt_q2-26', 'adhb_pkp_q1-26', 'adhb_pkp_q2-26', 'adhb_pmtb_q1-26', 'adhb_pmtb_q2-26', 'adhb_pi_q1-26', 'adhb_pi_q2-26', 'adhb_net_ekspor_q1-26', 'adhb_net_ekspor_q2-26', 'adhk_pdrb_q1-26', 'adhk_pdrb_q2-26', 'adhk_pkrt_q1-26', 'adhk_pkrt_q2-26', 'adhk_pklnprt_q1-26', 'adhk_pklnprt_q2-26', 'adhk_pkp_q1-26', 'adhk_pkp_q2-26', 'adhk_pmtb_q1-26', 'adhk_pmtb_q2-26', 'adhk_pi_q1-26', 'adhk_pi_q2-26', 'adhk_net_ekspor_q1-26', 'adhk_net_ekspor_q2-26', 'distribusi_pdrb_q1-26_p', 'distribusi_pdrb_q1-26_k', 'distribusi_pdrb_q2-26_p', 'distribusi_pdrb_q2-26_k', 'distribusi_pkrt_q1-26_p', 'distribusi_pkrt_q1-26_k', 'distribusi_pkrt_q2-26_p', 'distribusi_pkrt_q2-26_k', 'distribusi_pklnprt_q1-26_p', 'distribusi_pklnprt_q1-26_k', 'distribusi_pklnprt_q2-26_p', 'distribusi_pklnprt_q2-26_k', 'distribusi_pkp_q1-26_p', 'distribusi_pkp_q1-26_k', 'distribusi_pkp_q2-26_p', 'distribu

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [ ]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2026
            pdrb_tresh = 3
            if any(q in col for q in ["q1", "q2"]):
                if pd.notna(val) and (val < -pdrb_tresh or val > pdrb_tresh):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = f"Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-{pdrb_tresh}%)"
                    elif "adhk" in col:
                        evaluasi_text = f"Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-{pdrb_tresh}%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # # ADHB dan ADHK Q3 2025
            # elif "q3" in col:
            #     if pd.notna(val) and (val < -3 or val > 3):
            #         # Pisahkan nama kolom jadi komponen
            #         # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
            #         parts = col.split("_")
                    
            #         periode = parts[-1]  # ambil q1 / q2

            #         # tentukan evaluasi berdasarkan isi col
            #         if "adhb" in col:
            #             evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
            #         elif "adhk" in col:
            #             evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

            #         record = {
            #             "kode": df_clean.at[idx, "kode"],
            #             "provinsi": df_clean.at[idx, "provinsi"],
            #             "periode": periode,
            #             "nilai": val,
            #             "kolom": col,  # tambahan biar tau sumbernya
            #             "evaluasi": evaluasi_text
            #         }
            #         evaluasi_diskrepansi.append(record)
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1 dan Q2 2026
            komp_tresh = 5
            if pd.notna(val) and (val < -komp_tresh or val > komp_tresh):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = f"Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-{komp_tresh}%)"
                elif "adhk" in col:
                    evaluasi_text = f"Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-{komp_tresh}%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,  # tambahan biar tau sumbernya
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

    kode           provinsi periode      nilai                  kolom  \
0     91        Papua Barat   q2-26  -8.971973     adhb_pklnprt_q2-26   
1     82       Maluku Utara   q2-26  21.567658         adhb_pkp_q2-26   
2     92   Papua Barat Daya   q1-26  -6.022032        adhb_pmtb_q1-26   
3     82       Maluku Utara   q2-26  -8.480315        adhb_pmtb_q2-26   
4     92   Papua Barat Daya   q2-26  -5.130416        adhb_pmtb_q2-26   
..   ...                ...     ...        ...                    ...   
84    62  Kalimantan Tengah   q2-26   7.264897  adhk_net_ekspor_q2-26   
85    73   Sulawesi Selatan   q2-26  54.020387  adhk_net_ekspor_q2-26   
86    76     Sulawesi Barat   q2-26  -5.894661  adhk_net_ekspor_q2-26   
87    82       Maluku Utara   q2-26  17.918306  adhk_net_ekspor_q2-26   
88    92   Papua Barat Daya   q2-26  16.844285  adhk_net_ekspor_q2-26   

                                             evaluasi  
0   Diskrepansi ADHB antara Provinsi dan total Kab...  
1   Diskrep

#### 3. Evaluasi distribusi

In [239]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = re.sub(r'_[pk]$', '', col)
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

dist_tresh = 5

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_p")][0]
        col1 = [c for c in cols if c.endswith("_k")][0]
        
        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1 dan Q2 2026
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    }
                    evaluasi_dist.append(record)
                elif selisih > dist_tresh:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": f"Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-{dist_tresh} point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

   kode          provinsi periode      nilai  \
0    61  Kalimantan Barat   q2-26  -0.246995   
1    82      Maluku Utara   q2-26   0.016158   
2    91       Papua Barat   q2-26   1.495462   
3    92  Papua Barat Daya   q2-26  -0.066095   
4    33       Jawa Tengah   q2-26  -1.206847   
5    97  Papua Pegunungan   q2-26 -57.662813   

                                               kolom  \
0     distribusi_pi_q2-26_p vs distribusi_pi_q2-26_k   
1     distribusi_pi_q2-26_p vs distribusi_pi_q2-26_k   
2     distribusi_pi_q2-26_p vs distribusi_pi_q2-26_k   
3     distribusi_pi_q2-26_p vs distribusi_pi_q2-26_k   
4  distribusi_net_ekspor_q2-26_p vs distribusi_ne...   
5  distribusi_net_ekspor_q2-26_p vs distribusi_ne...   

                                            evaluasi  
0  Nilai antara Provinsi dan total Kabupaten/Kota...  
1  Nilai antara Provinsi dan total Kabupaten/Kota...  
2  Nilai antara Provinsi dan total Kabupaten/Kota...  
3  Nilai antara Provinsi dan total Kabupaten/Kota.

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [240]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = re.sub(r'_[pk]$', '', col)
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_p")][0]
        col1 = [c for c in cols if c.endswith("_k")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                qtoq_tresh = 1
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > qtoq_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{qtoq_tresh} point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > qtoq_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{qtoq_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                yony_tresh = 0.01
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2026
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > yony_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": f"Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{yony_tresh} point)"
                            }
                            evaluasi_growth.append(record)
                    # # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.01:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2026
                    yony_komp_tresh = 0.5
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > yony_komp_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": f"Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{yony_komp_tresh} point)"
                            }
                            evaluasi_growth.append(record)
                    # # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.5:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                ctoc_tresh = 1
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > ctoc_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-{ctoc_tresh} point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > ctoc_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-{ctoc_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                imp_qtoq_tresh = 1
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > imp_qtoq_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-{imp_qtoq_tresh} point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                imp_yony_tresh = 1
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > imp_yony_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-{imp_yony_tresh} point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

     kode          provinsi periode     nilai  \
0      11              Aceh   q1-26  0.167505   
1      76    Sulawesi Barat   q1-26 -4.370992   
2      92  Papua Barat Daya   q1-26 -0.169589   
3      76    Sulawesi Barat   q2-26  5.450999   
4      82      Maluku Utara   q2-26  5.740983   
..    ...               ...     ...       ...   
352    21    Kepulauan Riau   q2-26  7.426509   
353    61  Kalimantan Barat   q2-26  5.949107   
354    76    Sulawesi Barat   q2-26  3.869067   
355    82      Maluku Utara   q2-26  6.782755   
356    91       Papua Barat   q2-26  4.912101   

                                                 kolom  \
0           q-to-q_pdrb_q1-26_p vs q-to-q_pdrb_q1-26_k   
1           q-to-q_pdrb_q1-26_p vs q-to-q_pdrb_q1-26_k   
2           q-to-q_pdrb_q1-26_p vs q-to-q_pdrb_q1-26_k   
3           q-to-q_pdrb_q2-26_p vs q-to-q_pdrb_q2-26_k   
4           q-to-q_pdrb_q2-26_p vs q-to-q_pdrb_q2-26_k   
..                                                 ...   
352  

## Menggabungkan seluruh hasil evaluasi

In [241]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

   kode          provinsi periode      nilai               kolom  \
0    91       Papua Barat   q2-26  -8.971973  adhb_pklnprt_q2-26   
1    82      Maluku Utara   q2-26  21.567658      adhb_pkp_q2-26   
2    92  Papua Barat Daya   q1-26  -6.022032     adhb_pmtb_q1-26   
3    82      Maluku Utara   q2-26  -8.480315     adhb_pmtb_q2-26   
4    92  Papua Barat Daya   q2-26  -5.130416     adhb_pmtb_q2-26   

                                            evaluasi  
0  Diskrepansi ADHB antara Provinsi dan total Kab...  
1  Diskrepansi ADHB antara Provinsi dan total Kab...  
2  Diskrepansi ADHB antara Provinsi dan total Kab...  
3  Diskrepansi ADHB antara Provinsi dan total Kab...  
4  Diskrepansi ADHB antara Provinsi dan total Kab...  


## Menuliskan hasil evaluasi kedalam file excel

In [242]:
write_to_excel(evaluasi, "provinsi")

File baru dibuat: data/output/evaluasi_91.xlsx, dengan sheet 9100
File baru dibuat: data/output/evaluasi_82.xlsx, dengan sheet 8200
File baru dibuat: data/output/evaluasi_92.xlsx, dengan sheet 9200
File baru dibuat: data/output/evaluasi_73.xlsx, dengan sheet 7300
File baru dibuat: data/output/evaluasi_94.xlsx, dengan sheet 9400
File baru dibuat: data/output/evaluasi_11.xlsx, dengan sheet 1100
File baru dibuat: data/output/evaluasi_15.xlsx, dengan sheet 1500
File baru dibuat: data/output/evaluasi_18.xlsx, dengan sheet 1800
File baru dibuat: data/output/evaluasi_32.xlsx, dengan sheet 3200
File baru dibuat: data/output/evaluasi_52.xlsx, dengan sheet 5200
File baru dibuat: data/output/evaluasi_61.xlsx, dengan sheet 6100
File baru dibuat: data/output/evaluasi_76.xlsx, dengan sheet 7600
File baru dibuat: data/output/evaluasi_21.xlsx, dengan sheet 2100
File baru dibuat: data/output/evaluasi_31.xlsx, dengan sheet 3100
File baru dibuat: data/output/evaluasi_33.xlsx, dengan sheet 3300
File baru 

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [243]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Revisi & Growth")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-26_ril', 'adhb_pdrb_q1-26_rev', 'adhb_pdrb_q2-26', 'adhb_pkrt_q1-26_ril', 'adhb_pkrt_q1-26_rev', 'adhb_pkrt_q2-26', 'adhb_pklnprt_q1-26_ril', 'adhb_pklnprt_q1-26_rev', 'adhb_pklnprt_q2-26', 'adhb_pkp_q1-26_ril', 'adhb_pkp_q1-26_rev', 'adhb_pkp_q2-26', 'adhb_pmtb_q1-26_ril', 'adhb_pmtb_q1-26_rev', 'adhb_pmtb_q2-26', 'adhb_pi_q1-26_ril', 'adhb_pi_q1-26_rev', 'adhb_pi_q2-26', 'adhb_net_ekspor_q1-26_ril', 'adhb_net_ekspor_q1-26_rev', 'adhb_net_ekspor_q2-26', 'adhk_pdrb_q1-26_ril', 'adhk_pdrb_q1-26_rev', 'adhk_pdrb_q2-26', 'adhk_pkrt_q1-26_ril', 'adhk_pkrt_q1-26_rev', 'adhk_pkrt_q2-26', 'adhk_pklnprt_q1-26_ril', 'adhk_pklnprt_q1-26_rev', 'adhk_pklnprt_q2-26', 'adhk_pkp_q1-26_ril', 'adhk_pkp_q1-26_rev', 'adhk_pkp_q2-26', 'adhk_pmtb_q1-26_ril', 'adhk_pmtb_q1-26_rev', 'adhk_pmtb_q2-26', 'adhk_pi_q1-26_ril', 'adhk_pi_q1-26_rev', 'adhk_pi_q2-26', 'adhk_net_ekspor_q1-26_ril', 'adhk_net_ekspor_q1-26_rev', 'adhk_net_ekspor_q2-26', 'distribusi_pdrb_q1-26_ril'

#### 2. Evaluasi revisi ADHB dan ADHK

In [244]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

harga_tresh = 0.3

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > harga_tresh:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": f"Nilai revisi lebih dari (+-{harga_tresh * 100}%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

    kode                kabupaten/kota periode          nilai  \
0   3279                   Kota Banjar   q1-26  279938.352719   
1   6111        Kabupaten Kayong Utara   q1-26  811024.056585   
2   1302     Kabupaten Pesisir Selatan   q1-26   -4743.833067   
3   3404              Kabupaten Sleman   q1-26   60750.398681   
4   6502            Kabupaten Bulungan   q1-26  214475.000000   
5   7306                Kabupaten Gowa   q1-26   10466.806625   
6   7311                Kabupaten Bone   q1-26  375251.175731   
7   9704            Kabupaten Tolikara   q1-26   -1354.752773   
8   9706              Kabupaten Yalimo   q1-26     888.469658   
9   9708  Kabupaten Pegunungan Bintang   q1-26   -1160.549005   
10  1610           Kabupaten Ogan Ilir   q1-26   23739.264749   
11  3203             Kabupaten Cianjur   q1-26 -106411.623636   
12  3279                   Kota Banjar   q1-26 -750470.681592   
13  3376                    Kota Tegal   q1-26    9728.756101   
14  6111        Kabupaten

#### 3. Evaluasi revisi distribusi

In [245]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

revisi_dist_tresh = 5

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif (selisih > 5 and 'pi' not in col0 and 'net_ekspor' not in col0):
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": f"Nilai revisi lebih dari (+-{revisi_dist_tresh} point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

   kode          kabupaten/kota periode      nilai  \
0  3279             Kota Banjar   q1-26  18.313901   
1  6111  Kabupaten Kayong Utara   q1-26  48.522806   
2  9502  Kabupaten Boven Digoel   q1-26  61.017206   

                                               kolom  \
0  distribusi_pkp_q1-26_ril vs distribusi_pkp_q1-...   
1  distribusi_pmtb_q1-26_ril vs distribusi_pmtb_q...   
2  distribusi_net_ekspor_q1-26_ril vs distribusi_...   

                                            evaluasi  
0  Nilai revisi lebih dari (+-5 point) dari nilai...  
1  Nilai revisi lebih dari (+-5 point) dari nilai...  
2             Nilai revisi dan nilai rilis beda arah  


#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [246]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                growth_pdrb_tresh = 3
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > growth_pdrb_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Nilai revisi lebih dari (+-{growth_pdrb_tresh} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                growth_komp_tresh = 4
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > growth_komp_tresh:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": f"Nilai revisi lebih dari (+-{growth_komp_tresh} point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                sog_tresh = 2
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        if v1 < -sog_tresh or v1 > sog_tresh:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": f"SoG PI lebih dari +-{sog_tresh} persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

     kode                kabupaten/kota periode      nilai  \
0    6502            Kabupaten Bulungan   q1-26  -0.027106   
1    1105          Kabupaten Aceh Timur   q1-26   2.831825   
2    1407          Kabupaten Rokan Hulu   q1-26   6.780534   
3    3402              Kabupaten Bantul   q1-26   0.375204   
4    1409         Kabupaten Rokan Hilir   q1-26  -1.347320   
..    ...                           ...     ...        ...   
178  9601              Kabupaten Mimika   q1-26 -14.242341   
179  9708  Kabupaten Pegunungan Bintang   q1-26   0.053918   
180  1306     Kabupaten Padang Pariaman   q1-26  -0.105658   
181  1505         Kabupaten Muaro Jambi   q1-26  -0.014111   
182  3471               Kota Yogyakarta   q1-26   0.278829   

                                                 kolom  \
0       q-to-q_pkrt_q1-26_ril vs q-to-q_pkrt_q1-26_rev   
1    q-to-q_pklnprt_q1-26_ril vs q-to-q_pklnprt_q1-...   
2    q-to-q_pklnprt_q1-26_ril vs q-to-q_pklnprt_q1-...   
3    q-to-q_pklnprt_q1-

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [247]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kabkot" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kabkot dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kabkot")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            if pd.notna(val) and (val < -2 or val > 2):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,
                    "evaluasi": "SoG PI lebih dari +-2 persen"
                }
                evaluasi_value_growth.append(record)

# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

sog_q_pi_q2-26 44 2.1199865941104
sog_q_pi_q2-26 75 2.577162135176133
sog_q_pi_q2-26 85 3.6969846942946556
sog_q_pi_q2-26 132 2.3171339976608656
sog_q_pi_q2-26 136 4.393344491426281
sog_q_pi_q2-26 138 -2.65189692285704
sog_q_pi_q2-26 288 15.567816219333668
sog_q_pi_q2-26 292 2.2187032556878883
sog_q_pi_q2-26 293 5.737423708195275
sog_q_pi_q2-26 295 4.538917906777929
sog_q_pi_q2-26 296 5.658451916618465
sog_q_pi_q2-26 297 8.060172709942039
sog_q_pi_q2-26 298 4.449125770026006
sog_q_pi_q2-26 299 2.4616897802773257
sog_q_pi_q2-26 301 3.10015867679086
sog_q_pi_q2-26 302 3.813503275487357
sog_q_pi_q2-26 304 5.6161574824206415
sog_q_pi_q2-26 305 7.017967748896632
sog_q_pi_q2-26 306 2.699188995398659
sog_q_pi_q2-26 307 6.211179836967293
sog_q_pi_q2-26 308 3.7138953631669533
sog_q_pi_q2-26 309 6.1078956191226315
sog_q_pi_q2-26 310 3.6696487667227182
sog_q_pi_q2-26 312 5.74034676379973
sog_q_pi_q2-26 313 3.2968757177018944
sog_q_pi_q2-26 366 3.8923591342096753
sog_q_pi_q2-26 411 -2.769092121595

## Menggabungkan seluruh hasil evaluasi

In [248]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

   kode             kabupaten/kota periode          nilai  \
0  3279                Kota Banjar   q1-26  279938.352719   
1  6111     Kabupaten Kayong Utara   q1-26  811024.056585   
2  1302  Kabupaten Pesisir Selatan   q1-26   -4743.833067   
3  3404           Kabupaten Sleman   q1-26   60750.398681   
4  6502         Kabupaten Bulungan   q1-26  214475.000000   

                                        kolom  \
0    adhb_pkp_q1-26_ril vs adhb_pkp_q1-26_rev   
1  adhb_pmtb_q1-26_ril vs adhb_pmtb_q1-26_rev   
2      adhb_pi_q1-26_ril vs adhb_pi_q1-26_rev   
3      adhb_pi_q1-26_ril vs adhb_pi_q1-26_rev   
4      adhb_pi_q1-26_ril vs adhb_pi_q1-26_rev   

                                            evaluasi  
0  Nilai revisi lebih dari (+-30.0%) dari nilai r...  
1  Nilai revisi lebih dari (+-30.0%) dari nilai r...  
2  Nilai revisi lebih dari (+-30.0%) dari nilai r...  
3  Nilai revisi lebih dari (+-30.0%) dari nilai r...  
4  Nilai revisi lebih dari (+-30.0%) dari nilai r...  


## Menuliskan hasil evaluasi kedalam file excel

In [249]:
write_to_excel(evaluasi, "kabupaten/kota")

Sheet 3279 ditambahkan ke data/output/evaluasi_32.xlsx
Sheet 6111 ditambahkan ke data/output/evaluasi_61.xlsx
Sheet 1302 ditambahkan ke data/output/evaluasi_13.xlsx
Sheet 3404 ditambahkan ke data/output/evaluasi_34.xlsx
Sheet 6502 ditambahkan ke data/output/evaluasi_65.xlsx
Sheet 7306 ditambahkan ke data/output/evaluasi_73.xlsx
Sheet 7311 ditambahkan ke data/output/evaluasi_73.xlsx
Sheet 9704 ditambahkan ke data/output/evaluasi_97.xlsx
Sheet 9706 ditambahkan ke data/output/evaluasi_97.xlsx
Sheet 9708 ditambahkan ke data/output/evaluasi_97.xlsx
Sheet 1610 ditambahkan ke data/output/evaluasi_16.xlsx
Sheet 3203 ditambahkan ke data/output/evaluasi_32.xlsx
Sheet 3376 ditambahkan ke data/output/evaluasi_33.xlsx
Sheet 9502 ditambahkan ke data/output/evaluasi_95.xlsx
Sheet 5271 ditambahkan ke data/output/evaluasi_52.xlsx
Sheet 1606 ditambahkan ke data/output/evaluasi_16.xlsx
Sheet 1105 ditambahkan ke data/output/evaluasi_11.xlsx
Sheet 1407 ditambahkan ke data/output/evaluasi_14.xlsx
Sheet 3402